# Run ALL experiments (ablations + objective study)

**One click → everything runs and saves automatically.** This notebook trains every row
of the experiment matrix in sequence. Each `train()` already auto-runs the backtest +
figures, so when a run finishes its metrics are in `<out_dir>/backtest_summary.csv`. The
last cell scrapes every run and rewrites `experiments/results_master.csv` **and** the
paste-ready `manuals/04_results/EXPERIMENTS_TABLE.md`.

**Resumable + idempotent:** a run that already has a `backtest_summary.csv` is skipped,
so you can stop/restart and re-run this notebook any time. All the logic lives in
`experiments/experiment_runner.py` (so it's testable) — this notebook just drives it.

Runs trained (out dir = `experiments/runs_ablation/<run id>`):
`h30_tagc_post` · `h30_tagc_pre` · `h30_no_graph` (RQ1) · `h30_static_graph` (RQ2) ·
`h30_no_gru` (RQ3) · `h30_random_graph` (RQ4) · `h30_no_gat` · `obj_huber` · `obj_listmle`.

## 1. Setup (run from anywhere — finds the repo root)

In [ ]:
import os, sys, logging
# this notebook lives in experiments/ — walk up to the repo root (has model/ + experiments/)
for _ in range(6):
    if os.path.isdir('model') and os.path.isdir('experiments'):
        break
    os.chdir('..')
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), 'experiments'))
logging.basicConfig(level=logging.INFO, format='%(message)s')
import experiment_runner as er
print('repo root:', os.getcwd())
print('runs:', [r['run_id'] for r in er.RUNS])

## 2. Knobs

- **Full testing (default):** `MIN_EPOCHS = 12`, `MAX_EPOCHS = None` → each run does at
  least 12 epochs and keeps going while val Rank-IC improves (config cap 80, patience 10).
  ~1 h+ per run on CPU. For a fast pipeline check set `MIN_EPOCHS = MAX_EPOCHS = 5`.
- `DEVICE = 'cpu'` (or `'cuda'` / `'mps'` if available).
- `TARGET` only affects the single-stock point metrics, **not** the cross-sectional
  ranking the table reports — leave it unless you need a specific name.

Outputs are **saved after every run**, so a crash in a later run keeps the earlier results.

In [ ]:
# Full testing: at least MIN_EPOCHS per run, early-stopped above it (config cap = 80).
MIN_EPOCHS = 12      # epoch floor for every run (set MIN=MAX=5 for a quick smoke)
MAX_EPOCHS = None    # None = keep the config cap (80); early stopping decides when to stop
DEVICE = 'cpu'
TARGET = 'AAPL'

# preview the matrix before committing the time
import pandas as pd
pd.DataFrame(er.RUNS)[['run_id','variant','loss','period','notes']]

## 3. Run everything (the slow cell)

Trains each run in order, skipping any already finished. Safe to interrupt and re-run —
it picks up where it left off. Returns the collected results DataFrame.

In [ ]:
df = er.run_all(device=DEVICE, target=TARGET,
                min_epochs=MIN_EPOCHS, max_epochs=MAX_EPOCHS, skip_done=True)
df

## 4. Results (auto-saved)

`run_all` already wrote both files. Re-run just this cell any time to refresh the table
from whatever runs have finished (no retraining).

In [ ]:
df = er.collect()
print('saved ->', er.CSV_PATH)
print('saved ->', er.MD_PATH)
df